# Lifecycle Workflow

Simplified three-asset lifecycle simulation using Euro_Staat, Euro_ILBs, and Aandelen.

In [ ]:
import sys
from functools import partial
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from configs.ultimo_2025 import SCENARIO_CONFIG, LIFECYCLE_ASSETS
from asset_optimizer.data.loader import load_scenario_set
from asset_optimizer.lifecycle.rule_based_lifecycle import ASSET_COLUMNS, load_lifecycle_table, regime_for_state, rule_based_lifecycle
from asset_optimizer.lifecycle.simulation import run_lifecycle_simulation

## 1. Load Data

In [ ]:
scenario_set = load_scenario_set(SCENARIO_CONFIG)
print("Asset returns:", scenario_set.asset_returns.shape)
print("Yields:", scenario_set.yields.shape)
print("Lifecycle assets:", LIFECYCLE_ASSETS)

## 2. Run Lifecycle Simulation

In [ ]:
current_age = 45
retirement_age = 68
start_capital = 100.0
lifecycle_table = load_lifecycle_table(PROJECT_ROOT / "data" / "lifecycle_tables.csv")
policy = partial(rule_based_lifecycle, table=lifecycle_table)


normal_final_capital = run_lifecycle_simulation(
    scenario_set,
    current_age=current_age,
    retirement_age=retirement_age,
    start_capital=start_capital,
    annual_contribution=0.0,
    lifecycle_assets=LIFECYCLE_ASSETS,
    benchmark_name="Inflatie",
    policy=policy,
    inflation_name="Inflatie",
    rate_tenor=10,
)

years = min(retirement_age - current_age, scenario_set.horizon_years)
real_returns = (normal_final_capital / start_capital) ** (1.0 / years) - 1.0

print("Years simulated:", years)
print("Mean real return:", float(np.mean(real_returns)))
print("Real return percentiles:", np.percentile(real_returns, [5, 50, 95]))

## 3. Neutral vs Normal Protocol


In [ ]:
def neutral_policy(previous_returns, age, previous_interest_rate, previous_inflation):
    age = min(max(age, int(lifecycle_table["age"].min())), int(lifecycle_table["age"].max()))
    row = lifecycle_table[(lifecycle_table["lifecycle"] == "neutraal") & (lifecycle_table["age"] == age)]
    assert len(row) == 1
    return row.iloc[0][ASSET_COLUMNS].to_numpy(dtype=float).tolist()

neutral_final_capital = run_lifecycle_simulation(
    scenario_set,
    current_age=current_age,
    retirement_age=retirement_age,
    start_capital=start_capital,
    annual_contribution=0.0,
    lifecycle_assets=LIFECYCLE_ASSETS,
    benchmark_name="Inflatie",
    policy=neutral_policy,
    inflation_name="Inflatie",
    rate_tenor=10,
)

capital_distribution = pd.DataFrame({
    "neutral": neutral_final_capital,
    "normal_protocol": normal_final_capital,
})

capital_distribution.describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).T


In [ ]:
bins = [0, 75, 90, 100, 110, 125, 150, 200, np.inf]
hist_rows = []

for name in capital_distribution.columns:
    counts, edges = np.histogram(capital_distribution[name], bins=bins)
    for left, right, count in zip(edges[:-1], edges[1:], counts):
        bucket = f"{left:.0f}-{right:.0f}" if np.isfinite(right) else f"{left:.0f}+"
        hist_rows.append({"protocol": name, "bucket": bucket, "count": count})

pd.DataFrame(hist_rows).pivot(index="bucket", columns="protocol", values="count")


## 4. Regime Usage Over Time


In [ ]:
rate_tenor = 10
rate_tenor_index = scenario_set.yield_tenors.index(rate_tenor)
inflation_index = scenario_set.asset_names.index("Inflatie")

regime_rows = []
for year_index in range(years):
    age = current_age + year_index
    if year_index == 0:
        previous_rates = scenario_set.yields[:, 0, rate_tenor_index]
        previous_inflation = np.zeros(scenario_set.m)
    else:
        previous_rates = scenario_set.yields[:, year_index - 1, rate_tenor_index]
        previous_inflation = scenario_set.asset_returns[:, year_index - 1, inflation_index]

    regimes = [
        regime_for_state(float(rate), float(infl))
        for rate, infl in zip(previous_rates, previous_inflation)
    ]
    counts = pd.Series(regimes).value_counts(normalize=True)
    for regime, share in counts.items():
        regime_rows.append({"year": year_index + 1, "age": age, "regime": regime, "share": share})

regime_usage = pd.DataFrame(regime_rows)
regime_usage.pivot_table(index=["year", "age"], columns="regime", values="share", fill_value=0.0)


## 5. Turnover


In [ ]:
def lifecycle_weights_over_time(policy):
    asset_indices = [scenario_set.asset_names.index(name) for name in LIFECYCLE_ASSETS]
    lifecycle_returns = scenario_set.asset_returns[:, :years, asset_indices]
    rate_tenor_index = scenario_set.yield_tenors.index(10)
    inflation_index = scenario_set.asset_names.index("Inflatie")
    weights = np.empty((scenario_set.m, years, len(LIFECYCLE_ASSETS)), dtype=float)

    for year_index in range(years):
        age = current_age + year_index
        for scenario_index in range(scenario_set.m):
            if year_index == 0:
                previous_returns = np.zeros(len(LIFECYCLE_ASSETS), dtype=float)
                previous_interest_rate = scenario_set.yields[scenario_index, 0, rate_tenor_index]
                previous_inflation = 0.0
            else:
                previous_returns = lifecycle_returns[scenario_index, year_index - 1, :]
                previous_interest_rate = scenario_set.yields[scenario_index, year_index - 1, rate_tenor_index]
                previous_inflation = scenario_set.asset_returns[scenario_index, year_index - 1, inflation_index]

            weights[scenario_index, year_index] = policy(
                previous_returns,
                age,
                float(previous_interest_rate),
                float(previous_inflation),
            )

    return weights


neutral_weights = lifecycle_weights_over_time(neutral_policy)
normal_weights = lifecycle_weights_over_time(policy)


In [ ]:
def one_way_turnover(weights):
    return 0.5 * np.abs(np.diff(weights, axis=1)).sum(axis=2)


def turnover_by_year(weights, protocol):
    turnover = one_way_turnover(weights)
    return pd.DataFrame({
        "protocol": protocol,
        "year": np.arange(2, years + 1),
        "age": np.arange(current_age + 1, current_age + years),
        "mean_turnover": turnover.mean(axis=0),
        "p50_turnover": np.percentile(turnover, 50, axis=0),
        "p95_turnover": np.percentile(turnover, 95, axis=0),
    })


turnover_summary = pd.concat([
    turnover_by_year(neutral_weights, "neutral"),
    turnover_by_year(normal_weights, "normal_protocol"),
], ignore_index=True)

turnover_summary


In [ ]:
total_turnover = pd.DataFrame({
    "neutral": one_way_turnover(neutral_weights).sum(axis=1),
    "normal_protocol": one_way_turnover(normal_weights).sum(axis=1),
})

total_turnover.describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).T
